# Flux LoRA Training — Colab T4

Train a custom LoRA of a specific person using **ai-toolkit** (Ostris) on a free T4 GPU.

**What you get:** A LoRA file (.safetensors) you can use in ComfyUI with the LoraLoader node to generate images of a person in any style/setting.

**Time:** ~30-60 min training on T4
**VRAM:** ~14 GB peak

**Run cells 1-5 in order.**

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install ai-toolkit + Dependencies
%cd /content
!git clone https://github.com/ostris/ai-toolkit.git 2>/dev/null || echo "Already cloned"
%cd /content/ai-toolkit
!git submodule update --init --recursive
!pip install -r requirements.txt -q
!pip install peft accelerate safetensors -q

print("\nai-toolkit installed!")

In [ ]:
#@title 3. Upload Training Images (15-30 photos)
#@markdown Upload 15-30 photos of the person. More variety = better results.
#@markdown - Different angles (front, 3/4, profile)
#@markdown - Different lighting (indoor, outdoor, studio)
#@markdown - Different expressions
#@markdown - Clear face, good quality

from google.colab import files
import os

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    path = os.path.join(DATASET_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"Saved: {path}")

count = len(os.listdir(DATASET_DIR))
print(f"\nTotal images: {count}")
if count < 10:
    print("WARNING: Less than 10 images. Recommended: 15-30 for best results.")

In [ ]:
#@title 4. Configure & Train LoRA
#@markdown Set your training parameters below.

trigger_word = "ohwx" #@param {type:"string"}
#@markdown Trigger word — a unique token that activates the LoRA. Use something rare like "ohwx" or "sks".

steps = 1500 #@param {type:"integer"}
#@markdown Training steps (1000-2000 recommended for T4)

learning_rate = 1e-4 #@param {type:"number"}
resolution = 512 #@param [512, 768] {type:"raw"}
lora_rank = 16 #@param [8, 16, 32] {type:"raw"}
output_name = "my_person_lora" #@param {type:"string"}

import yaml, os

config = {
    "job": "extension",
    "config": {
        "name": output_name,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "performance_log_every": 100,
            "device": "cuda:0",
            "trigger_word": trigger_word,
            "network": {
                "type": "lora",
                "linear": lora_rank,
                "linear_alpha": lora_rank
            },
            "save": {
                "dtype": "float16",
                "save_every": 500,
                "max_step_saves_to_keep": 2
            },
            "datasets": [{
                "folder_path": "/content/dataset",
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": [resolution, resolution]
            }],
            "train": {
                "batch_size": 1,
                "steps": steps,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": learning_rate,
                "ema_config": {"use_ema": True, "ema_decay": 0.99},
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": 500,
                "width": resolution,
                "height": resolution,
                "prompts": [
                    f"a photo of {trigger_word} person, professional headshot, studio lighting",
                    f"a photo of {trigger_word} person, casual outdoor portrait, natural light"
                ],
                "neg": "",
                "seed": 42,
                "walk_seed": True,
                "guidance_scale": 3.5,
                "sample_steps": 25
            }
        }]
    }
}

config_path = "/content/ai-toolkit/config/train_lora.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved to {config_path}")
print(f"\nTraining settings:")
print(f"  Trigger word: {trigger_word}")
print(f"  Steps: {steps}")
print(f"  Learning rate: {learning_rate}")
print(f"  Resolution: {resolution}x{resolution}")
print(f"  LoRA rank: {lora_rank}")
print(f"\nStarting training... (this takes 30-60 min on T4)")

!cd /content/ai-toolkit && python run.py config/train_lora.yaml

print("\nTraining complete!")

In [ ]:
#@title 5. Test LoRA — Generate a Test Image
#@markdown Quick test of the trained LoRA using the diffusers pipeline.

trigger_word = "ohwx" #@param {type:"string"}
prompt = "a photo of ohwx person, professional headshot, studio lighting, 4k" #@param {type:"string"}
output_name = "my_person_lora" #@param {type:"string"}

import glob

# Find the latest LoRA file
lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("No LoRA file found! Make sure training completed.")
else:
    lora_path = lora_files[-1]
    print(f"Using LoRA: {lora_path}")

    import torch
    from diffusers import FluxPipeline

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.bfloat16
    ).to("cuda")
    pipe.load_lora_weights(lora_path)

    image = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=3.5,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    image.save("/content/test_result.png")

    from IPython.display import display, Image
    display(Image("/content/test_result.png"))
    print("Test image saved to /content/test_result.png")

    del pipe
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
#@title 6. Download LoRA / Save to Google Drive
#@markdown Choose where to save your trained LoRA file.

save_to_drive = True #@param {type:"boolean"}
output_name = "my_person_lora" #@param {type:"string"}

import glob, shutil, os

lora_files = sorted(glob.glob(f"/content/output/{output_name}/*.safetensors"))
if not lora_files:
    print("No LoRA file found!")
else:
    lora_path = lora_files[-1]
    lora_filename = os.path.basename(lora_path)
    print(f"LoRA file: {lora_path} ({os.path.getsize(lora_path) / 1024**2:.1f} MB)")

    if save_to_drive:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dir = "/content/drive/MyDrive/ComfyUI_LoRAs"
        os.makedirs(drive_dir, exist_ok=True)
        dest = f"{drive_dir}/{lora_filename}"
        shutil.copy2(lora_path, dest)
        print(f"Saved to Drive: {dest}")
    else:
        from google.colab import files
        files.download(lora_path)
        print("Download started!")

---
## Using Your LoRA in ComfyUI

### Method 1: LoraLoader Node
1. Copy LoRA file to `ComfyUI/models/loras/`
2. In your workflow, add **LoraLoader** node
3. Connect it between the model loader and KSampler
4. Select your LoRA file
5. Use your trigger word in the prompt (e.g., "a photo of ohwx person in a garden")

### Method 2: In Colab
Use the "Download LoRA" cell in `colab_flux_photo.ipynb` to load it directly.

### Tips
- **Trigger word** must appear in your prompt for the LoRA to activate
- **Strength** 0.7-1.0 works best for face LoRAs
- **Lower strength** (0.4-0.6) for more stylistic variation
- Combine with IPAdapter FaceID for even better consistency

### Troubleshooting
- **Overfit** (all images look the same): Reduce steps, increase dataset variety
- **Underfit** (face not recognizable): Increase steps, improve image quality
- **OOM errors**: Reduce resolution to 512, reduce batch_size to 1